# Lab 2 · Clean, connect and check

**In class:** Part A, guided, on a messy version of the penguins data · **At home:** Part B, the same steps on your own two tables
**Due:** Sunday, November 8 at 11:59 PM (extended for Election Day) · **Submit:** your repo URL on CourseWorks

Tonight's three skills:
1. **Clean** a messy CSV with pandas and load it into DuckDB
2. **Connect** two tables with a join, on a key you've checked
3. **Check** every join by counting rows before and after

Run cells top to bottom with **Shift+Enter**.

**Name:** Mengjing Xu  
**AI assistance:** ChatGPT/Codex substantially assisted with code preparation, execution, checks and written explanations. Outputs are actual executed results. The official Part A template is retained; Part B uses the existing Lab 1 files.

---
# Part A · Guided: messy penguins + a lookup table

## A1. Read the messy file with pandas

`penguins_raw.csv` is the same study as last week's penguins, in its original form: long column names, extra fields, comments. **pandas** is a Python library for working with tables; a table in pandas is called a **DataFrame**.

In [1]:
from pathlib import Path
import pandas as pd
import duckdb

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
raw = pd.read_csv(ROOT / "data" / "penguins_raw.csv")

print(raw.shape)   # (rows, columns)
raw.head()

(344, 17)


,studyName,Sample Number,Species,Region,Island,Stage,Individual ID,Clutch Completion,Date Egg,Culmen Length (mm),Culmen Depth (mm),Flipper Length (mm),Body Mass (g),Sex,Delta 15 N (o/oo),Delta 13 C (o/oo),Comments
0,PAL0708,1,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N1A1,Yes,2007-11-11,39.1,18.7,181.0,3750.0,MALE,NaN,NaN,Not enough blood for isotopes.
1,PAL0708,2,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N1A2,Yes,2007-11-11,39.5,17.4,186.0,3800.0,FEMALE,8.94956,-24.69454,NaN
2,PAL0708,3,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N2A1,Yes,2007-11-16,40.3,18.0,195.0,3250.0,FEMALE,8.36821,-25.33302,NaN
3,PAL0708,4,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N2A2,Yes,2007-11-16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Adult not sampled.
4,PAL0708,5,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N3A1,Yes,2007-11-16,36.7,19.3,193.0,3450.0,FEMALE,8.76651,-25.32426,NaN


How much is missing in each column? `isna()` marks missing values; `.sum()` counts them per column.

In [2]:
raw.isna().sum()

studyName                0
Sample Number            0
Species                  0
Region                   0
Island                   0
Stage                    0
Individual ID            0
Clutch Completion        0
Date Egg                 0
Culmen Length (mm)       2
Culmen Depth (mm)        2
Flipper Length (mm)      2
Body Mass (g)            2
Sex                     11
Delta 15 N (o/oo)       14
Delta 13 C (o/oo)       13
Comments               290
dtype: int64

Look at the species column. What would make it hard to use as a key?

In [3]:
raw["Species"].value_counts()

Species
Adelie Penguin (Pygoscelis adeliae)          152
Gentoo penguin (Pygoscelis papua)            124
Chinstrap penguin (Pygoscelis antarctica)     68
Name: count, dtype: int64

**Notice:** long names with the Latin name in parentheses, and inconsistent capitals ("Adelie **P**enguin" vs. "Gentoo **p**enguin"). Text that *looks* the same to a person often won't match exactly in a join.

## A2. Three cleaning steps

**Step 1. Rename columns** to short, lowercase names with no spaces, so they're easy to type in SQL.

In [4]:
penguins = raw.rename(columns={
    "studyName": "study",
    "Individual ID": "individual_id",
    "Species": "species_raw",
    "Island": "island",
    "Date Egg": "date_egg",
    "Clutch Completion": "clutch_completion",
    "Culmen Length (mm)": "bill_length_mm",
    "Culmen Depth (mm)": "bill_depth_mm",
    "Flipper Length (mm)": "flipper_length_mm",
    "Body Mass (g)": "body_mass_g",
    "Sex": "sex",
})
penguins.columns.tolist()

['study',
 'Sample Number',
 'species_raw',
 'Region',
 'island',
 'Stage',
 'individual_id',
 'clutch_completion',
 'date_egg',
 'bill_length_mm',
 'bill_depth_mm',
 'flipper_length_mm',
 'body_mass_g',
 'sex',
 'Delta 15 N (o/oo)',
 'Delta 13 C (o/oo)',
 'Comments']

**Step 2. Standardize text.** Make `sex` lowercase, and pull the Latin name out of the parentheses into its own column. Scientific names are written the same way everywhere, which makes them a better key than the common name.

In [5]:
penguins["sex"] = penguins["sex"].str.lower()
penguins["latin_name"] = penguins["species_raw"].str.extract(r"\((.*)\)")   # text inside ( )

penguins[["species_raw", "latin_name", "sex"]].drop_duplicates().head(6)

,species_raw,latin_name,sex
0,Adelie Penguin (Pygoscelis adeliae),Pygoscelis adeliae,male
1,Adelie Penguin (Pygoscelis adeliae),Pygoscelis adeliae,female
3,Adelie Penguin (Pygoscelis adeliae),Pygoscelis adeliae,NaN
152,Gentoo penguin (Pygoscelis papua),Pygoscelis papua,female
153,Gentoo penguin (Pygoscelis papua),Pygoscelis papua,male
178,Gentoo penguin (Pygoscelis papua),Pygoscelis papua,NaN


**Step 3. Fix types and keep what you need.** The egg date came in as text; make it a real date. Then keep only the columns we'll use.

In [6]:
penguins["date_egg"] = pd.to_datetime(penguins["date_egg"])

penguins = penguins[[
    "study", "individual_id", "latin_name", "island", "date_egg", "clutch_completion",
    "bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g", "sex",
]]
penguins.dtypes

study                        object
individual_id                object
latin_name                   object
island                       object
date_egg             datetime64[ns]
clutch_completion            object
bill_length_mm              float64
bill_depth_mm               float64
flipper_length_mm           float64
body_mass_g                 float64
sex                          object
dtype: object

## A3. Load it into DuckDB and verify with SQL

DuckDB can read a pandas DataFrame directly by its variable name. Always check the row count after moving data: **rows in should equal rows out.**

In [7]:
con = duckdb.connect(str(ROOT / "lab2.duckdb"))
con.sql("CREATE OR REPLACE TABLE penguins_clean AS SELECT * FROM penguins")

print("rows in pandas:", len(penguins))
con.sql("SELECT count(*) AS rows_in_duckdb FROM penguins_clean")

rows in pandas: 344


┌────────────────┐
│ rows_in_duckdb │
│     int64      │
├────────────────┤
│            344 │
└────────────────┘

In [8]:
con.sql("DESCRIBE penguins_clean")

┌───────────────────┬──────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name    │ column_type  │  null   │   key   │ default │  extra  │
│      varchar      │   varchar    │ varchar │ varchar │ varchar │ varchar │
├───────────────────┼──────────────┼─────────┼─────────┼─────────┼─────────┤
│ study             │ VARCHAR      │ YES     │ NULL    │ NULL    │ NULL    │
│ individual_id     │ VARCHAR      │ YES     │ NULL    │ NULL    │ NULL    │
│ latin_name        │ VARCHAR      │ YES     │ NULL    │ NULL    │ NULL    │
│ island            │ VARCHAR      │ YES     │ NULL    │ NULL    │ NULL    │
│ date_egg          │ TIMESTAMP_NS │ YES     │ NULL    │ NULL    │ NULL    │
│ clutch_completion │ VARCHAR      │ YES     │ NULL    │ NULL    │ NULL    │
│ bill_length_mm    │ DOUBLE       │ YES     │ NULL    │ NULL    │ NULL    │
│ bill_depth_mm     │ DOUBLE       │ YES     │ NULL    │ NULL    │ NULL    │
│ flipper_length_mm │ DOUBLE       │ YES     │ NULL    │ NULL    │ NULL    │

## A4. What is one row? Is the ID really an ID?

There's a column called `individual_id`. If it identified one penguin, every penguin would appear once per season. Let's check.

In [9]:
con.sql("""
    SELECT count(*)                      AS rows,
           count(DISTINCT individual_id) AS distinct_ids
    FROM penguins_clean
""")

┌───────┬──────────────┐
│ rows  │ distinct_ids │
│ int64 │    int64     │
├───────┼──────────────┤
│   344 │          190 │
└───────┴──────────────┘

In [10]:
con.sql("""
    SELECT study, latin_name, island, individual_id
    FROM penguins_clean
    WHERE individual_id = 'N11A1'
""")

┌─────────┬────────────────────┬─────────┬───────────────┐
│  study  │     latin_name     │ island  │ individual_id │
│ varchar │      varchar       │ varchar │    varchar    │
├─────────┼────────────────────┼─────────┼───────────────┤
│ PAL0708 │ Pygoscelis adeliae │ Biscoe  │ N11A1         │
│ PAL0809 │ Pygoscelis papua   │ Biscoe  │ N11A1         │
└─────────┴────────────────────┴─────────┴───────────────┘

**The same ID belongs to an Adélie penguin in one season and a Gentoo the next.** It can't be one bird. It's most likely a nest-and-adult label the researchers reused. So:

- `individual_id` **alone is not a key.** A column named "ID" isn't automatically unique; always check with `count(DISTINCT ...)`.
- **One row = one adult penguin at one nest in one breeding season.** That's the table's **grain**, and you should be able to say it for every table you use.

## A5. A lookup table to join

`species_lookup.csv` is a small reference table: one row per species, with its common name, Latin name, genus and group.

In [11]:
con.sql(f"CREATE OR REPLACE TABLE species AS SELECT * FROM read_csv('{ROOT / 'data' / 'species_lookup.csv'}')")
con.sql("SELECT * FROM species")

┌───────────────────┬────────────────────────┬─────────────┬───────────────────────┐
│    common_name    │       latin_name       │    genus    │      group_name       │
│      varchar      │        varchar         │   varchar   │        varchar        │
├───────────────────┼────────────────────────┼─────────────┼───────────────────────┤
│ Adélie penguin    │ Pygoscelis adeliae     │ Pygoscelis  │ brush-tailed penguins │
│ Chinstrap penguin │ Pygoscelis antarcticus │ Pygoscelis  │ brush-tailed penguins │
│ Gentoo penguin    │ Pygoscelis papua       │ Pygoscelis  │ brush-tailed penguins │
│ Emperor penguin   │ Aptenodytes forsteri   │ Aptenodytes │ great penguins        │
└───────────────────┴────────────────────────┴─────────────┴───────────────────────┘

Before joining, check that the key is **unique** in the lookup table. If a key appeared twice, every matching penguin would be copied twice.

In [12]:
con.sql("""
    SELECT latin_name, count(*) AS n
    FROM species
    GROUP BY latin_name
    HAVING count(*) > 1
""")   # no rows back = the key is unique

┌────────────┬───────┐
│ latin_name │   n   │
│  varchar   │ int64 │
├────────────┴───────┤
│       0 rows       │
└────────────────────┘

## A6. Join, and count what comes out

`JOIN ... ON` lines up rows where the key matches. We expect **344** rows out, one per penguin row.

In [13]:
con.sql("""
    SELECT count(*) AS rows_after_join
    FROM penguins_clean p
    JOIN species s ON p.latin_name = s.latin_name
""")

┌─────────────────┐
│ rows_after_join │
│      int64      │
├─────────────────┤
│             276 │
└─────────────────┘

**Only 276, not 344.** An inner `JOIN` silently drops rows that don't match. Nothing errored. This is a data cascade in the making.

## A7. Find what didn't match with a LEFT JOIN

A `LEFT JOIN` keeps every penguin row. Where there's no match, the lookup's columns come back empty (`NULL`), so we can find the problem rows.

In [14]:
con.sql("""
    SELECT p.latin_name, count(*) AS unmatched_rows
    FROM penguins_clean p
    LEFT JOIN species s ON p.latin_name = s.latin_name
    WHERE s.latin_name IS NULL
    GROUP BY p.latin_name
""")

┌───────────────────────┬────────────────┐
│      latin_name       │ unmatched_rows │
│        varchar        │     int64      │
├───────────────────────┼────────────────┤
│ Pygoscelis antarctica │             68 │
└───────────────────────┴────────────────┘

Compare with the lookup: the data says **Pygoscelis antarctic*a***, the lookup says **Pygoscelis antarctic*us***. Two sources spelled the same species differently, and 68 penguins vanished from the join.

## A8. Fix the key, then join again

Fix it at the source of the problem (the cleaning step), then reload and rejoin.

In [15]:
penguins["latin_name"] = penguins["latin_name"].replace(
    {"Pygoscelis antarctica": "Pygoscelis antarcticus"}
)
con.sql("CREATE OR REPLACE TABLE penguins_clean AS SELECT * FROM penguins")

con.sql("""
    SELECT count(*) AS rows_after_join
    FROM penguins_clean p
    JOIN species s ON p.latin_name = s.latin_name
""")   # now 344

┌─────────────────┐
│ rows_after_join │
│      int64      │
├─────────────────┤
│             344 │
└─────────────────┘

Which way you join matters. Start from the **lookup** with a `LEFT JOIN` to see species that have no penguins:

In [16]:
con.sql("""
    SELECT s.common_name, count(p.latin_name) AS penguin_rows
    FROM species s
    LEFT JOIN penguins_clean p ON p.latin_name = s.latin_name
    GROUP BY s.common_name
    ORDER BY penguin_rows DESC
""")

┌───────────────────┬──────────────┐
│    common_name    │ penguin_rows │
│      varchar      │    int64     │
├───────────────────┼──────────────┤
│ Adélie penguin    │          152 │
│ Gentoo penguin    │          124 │
│ Chinstrap penguin │           68 │
│ Emperor penguin   │            0 │
└───────────────────┴──────────────┘

## A9. What a duplicate key does

Here we fake a lookup table that lists Gentoo twice, then join. Watch the row count.

In [17]:
con.sql("""
    WITH species_with_dupe AS (
        SELECT * FROM species
        UNION ALL
        SELECT * FROM species WHERE common_name = 'Gentoo penguin'
    )
    SELECT count(*) AS rows_after_join
    FROM penguins_clean p
    JOIN species_with_dupe s ON p.latin_name = s.latin_name
""")   # 468: every Gentoo row was copied

┌─────────────────┐
│ rows_after_join │
│      int64      │
├─────────────────┤
│             468 │
└─────────────────┘

**344 became 468**, and every total or average involving Gentoos would now be wrong. That's why you check key uniqueness (A5) before every join.

## A10. Use the join to answer a question

Now that the tables connect, ask a question that needs both: the lookup's readable names, the penguin table's measurements.

In [18]:
con.sql("""
    SELECT s.common_name,
           p.island,
           count(*)                    AS penguins,
           round(avg(p.body_mass_g))   AS avg_mass_g
    FROM penguins_clean p
    JOIN species s ON p.latin_name = s.latin_name
    GROUP BY s.common_name, p.island
    ORDER BY s.common_name, p.island
""")

┌───────────────────┬───────────┬──────────┬────────────┐
│    common_name    │  island   │ penguins │ avg_mass_g │
│      varchar      │  varchar  │  int64   │   double   │
├───────────────────┼───────────┼──────────┼────────────┤
│ Adélie penguin    │ Biscoe    │       44 │     3710.0 │
│ Adélie penguin    │ Dream     │       56 │     3688.0 │
│ Adélie penguin    │ Torgersen │       52 │     3706.0 │
│ Chinstrap penguin │ Dream     │       68 │     3733.0 │
│ Gentoo penguin    │ Biscoe    │      124 │     5076.0 │
└───────────────────┴───────────┴──────────┴────────────┘

### The join checklist (use it in Part B)

1. **Grain:** what is one row in each table?
2. **Key:** which column connects them, and is it **unique** in the lookup side?
3. **Clean the key** the same way on both sides (spelling, capitals, spaces, codes vs. names, types).
4. **Count rows** before and after. More rows = a duplicate key. Fewer = unmatched keys.
5. **LEFT JOIN … IS NULL** to see exactly what didn't match, then decide: fix, drop, or keep as unknown.

---
# Part B · At home: 311 requests and population

**Question:** Which matched ZIP/ZCTA areas have the most September 2026 NYC311 service requests per 1,000 estimated residents?

**Why this question matters:** Comparing counts alone can favor populous areas. Combining request counts with population gives a different descriptive comparison, while also exposing cases where a small population denominator makes a rate misleading. The goal is a descriptive comparison of request counts and population, not a claim about personal experience.

## B1. Load both tables with pandas


In [19]:
FILE_A = "311_sep2026.csv"
FILE_B = "population_zcta2024.csv"

# Read identifiers as strings from the CSV: converting after numeric inference
# cannot reliably recover a ZIP code's leading zero.
a = pd.read_csv(ROOT / "data" / FILE_A, dtype={"unique_key": "string", "incident_zip": "string"})
b = pd.read_csv(ROOT / "data" / FILE_B, dtype={"zip_code": "string"})
raw_counts = {"complaints": len(a), "population": len(b)}
print("A:", a.shape, " B:", b.shape)
a.head()


A: (323630, 8)  B: (218, 4)


,unique_key,created_date,closed_date,agency,complaint_type,incident_zip,borough,status
0,70250274,2026-09-01T00:45:08.000,2026-09-08T16:50:00.000,DEP,Water Maintenance,10465,BRONX,In Progress
1,70250298,2026-09-01T00:06:18.000,2026-09-03T12:45:00.000,DEP,Sewer Maintenance,10025,MANHATTAN,In Progress
2,70250323,2026-09-01T00:03:12.000,2026-09-01T23:23:00.000,DEP,Noise,11209,BROOKLYN,Closed
3,70250325,2026-09-01T00:36:12.000,2026-09-02T21:00:00.000,DEP,Noise,10014,MANHATTAN,Closed
4,70250354,2026-09-01T01:59:11.000,2026-09-01T05:28:42.000,NYPD,Drug Activity,10456,BRONX,Closed


In [20]:
b.head()

,zip_code,population,population_moe,estimate_period
0,10001,30511,1797,2020-2024
1,10002,76873,3494,2020-2024
2,10003,53560,2107,2020-2024
3,10004,3711,909,2020-2024
4,10005,9439,1107,2020-2024


### Quality checks before cleaning
These checks target the rate calculation: request identity and date scope, missing/placeholder postal codes, lookup key uniqueness, zero or negative denominators, numeric types and categorical text. No rows are removed merely to make a join work.

In [21]:
display(a.dtypes.to_frame("A_type"), b.dtypes.to_frame("B_type"))
display(a.isna().sum().to_frame("A_missing"), b.isna().sum().to_frame("B_missing"))
print("Repeated full request rows:", a.duplicated().sum())
print("Repeated request IDs:", a["unique_key"].duplicated().sum())
print("Repeated lookup ZIPs:", b["zip_code"].duplicated().sum())
print("ZIP example values side by side:")
display(pd.DataFrame({"A_incident_zip": a["incident_zip"].dropna().drop_duplicates().head(10).reset_index(drop=True),
                      "B_zip_code": b["zip_code"].dropna().drop_duplicates().head(10).reset_index(drop=True)}))
print("Borough values:")
display(a["borough"].value_counts(dropna=False).to_frame("requests"))
print("Population range:", b["population"].min(), b["population"].max())
print("Negative population:", b["population"].lt(0).sum())
print("Zero population:", b["population"].eq(0).sum())
print("Estimate periods:", b["estimate_period"].value_counts().to_dict())
print("Created range:", a["created_date"].min(), a["created_date"].max())
print("Potential placeholder ZIP 10000:", a["incident_zip"].eq("10000").sum())


,A_type
unique_key,string[python]
created_date,object
closed_date,object
agency,object
complaint_type,object
incident_zip,string[python]
borough,object
status,object


,B_type
zip_code,string[python]
population,int64
population_moe,int64
estimate_period,object


,A_missing
unique_key,0
created_date,0
closed_date,25424
agency,0
complaint_type,0
incident_zip,2973
borough,0
status,0


,B_missing
zip_code,0
population,0
population_moe,0
estimate_period,0


Repeated full request rows: 0
Repeated request IDs: 0
Repeated lookup ZIPs: 0
ZIP example values side by side:


,A_incident_zip,B_zip_code
0,10465,10001
1,10025,10002
2,11209,10003
3,10014,10004
4,10456,10005
5,10472,10006
6,10453,10007
7,10457,10009
8,10460,10010
9,10468,10011


Borough values:


,requests
borough,
BROOKLYN,105705
QUEENS,80444
MANHATTAN,64994
BRONX,59213
STATEN ISLAND,12877
Unspecified,397


Population range: 0 105716
Negative population: 0
Zero population: 22
Estimate periods: {'2020-2024': 218}
Created range: 2026-09-01T00:00:05.000 2026-09-30T23:59:58.000
Potential placeholder ZIP 10000: 32


**Findings and decisions.** The 323,630 requests have no repeated IDs. There are 2,973 missing postal codes and 25,424 missing closure dates. The population lookup has 218 rows, no duplicate or missing postal codes, and 22 zero estimates; zero is retained as a source value and excluded only from division. Population and margin of error are numeric, and all estimate-period values are 2020–2024. The category Unspecified is not automatically treated as a known borough. The type conversion below also checks parsing and flags 213 closures earlier than creation, rather than using those records for response-time analysis. Our volume question uses creation time, so these closure anomalies do not justify deleting the request rows.

**Grain.** One row in A is one NYC311 service request created during September 2026, identified by unique_key. It is not necessarily one person or a unique real-world incident. One row in B is one selected Census ZCTA with its total population estimate and margin of error for the 2020–2024 ACS five-year period. B covers 218 ZCTAs selected from ZIP codes appearing in A; it is not a complete NYC area list.

## B2. Clean: at least three steps


In [22]:
# Step 1: normalize headers and rename both postal-code fields to a shared name.
a.columns = a.columns.str.strip().str.lower()
b.columns = b.columns.str.strip().str.lower()
a = a.rename(columns={"incident_zip": "zip_code"})

# Step 2: trim identifier text without replacing missing values with the word 'nan'.
for frame in (a, b):
    frame["zip_code"] = frame["zip_code"].astype("string").str.strip().replace("", pd.NA)
    present = frame["zip_code"].notna()
    assert frame.loc[present, "zip_code"].str.fullmatch(r"\d{5}").all(), "Review malformed ZIP/ZCTA codes"
a["borough"] = a["borough"].astype("string").str.strip().str.upper()

# Step 3: parse date/time columns and population measures into usable types.
# Keep source date strings so any parsing problems remain auditable.
for column in ("created_date", "closed_date"):
    original = a[column].copy()
    a[column] = pd.to_datetime(original, errors="coerce")
    assert not (original.notna() & a[column].isna()).any(), f"Unparsed {column}"
for column in ("population", "population_moe"):
    b[column] = pd.to_numeric(b[column], errors="raise").astype("Int64")

# Step 4: flag invalid event ordering rather than deleting the original records.
a["closed_before_created"] = a["closed_date"].notna() & (a["closed_date"] < a["created_date"])
assert len(a) == raw_counts["complaints"] and len(b) == raw_counts["population"]
assert a["unique_key"].notna().all() and a["unique_key"].is_unique
assert b["zip_code"].notna().all() and b["zip_code"].is_unique
assert b["population"].notna().all() and b["population"].ge(0).all()
print("Rows retained:", raw_counts)
print("Missing ZIPs:", a["zip_code"].isna().sum())
print("Closure before creation:", a["closed_before_created"].sum())
print("Zero-population areas:", b["population"].eq(0).sum())
display(a.dtypes.to_frame("type"), b.dtypes.to_frame("type"))
a.head()


Rows retained: {'complaints': 323630, 'population': 218}
Missing ZIPs: 2973
Closure before creation: 213
Zero-population areas: 22


,type
unique_key,string[python]
created_date,datetime64[ns]
closed_date,datetime64[ns]
agency,object
complaint_type,object
zip_code,string[python]
borough,string[python]
status,object
closed_before_created,bool


,type
zip_code,string[python]
population,Int64
population_moe,Int64
estimate_period,object


,unique_key,created_date,closed_date,agency,complaint_type,zip_code,borough,status,closed_before_created
0,70250274,2026-09-01 00:45:08,2026-09-08 16:50:00,DEP,Water Maintenance,10465,BRONX,In Progress,False
1,70250298,2026-09-01 00:06:18,2026-09-03 12:45:00,DEP,Sewer Maintenance,10025,MANHATTAN,In Progress,False
2,70250323,2026-09-01 00:03:12,2026-09-01 23:23:00,DEP,Noise,11209,BROOKLYN,Closed,False
3,70250325,2026-09-01 00:36:12,2026-09-02 21:00:00,DEP,Noise,10014,MANHATTAN,Closed,False
4,70250354,2026-09-01 01:59:11,2026-09-01 05:28:42,NYPD,Drug Activity,10456,BRONX,Closed,False


## B3. Load into DuckDB and check the key

In [23]:
con.sql("CREATE OR REPLACE TABLE table_a AS SELECT * FROM a")
con.sql("CREATE OR REPLACE TABLE table_b AS SELECT * FROM b")
KEY_A = "zip_code"
KEY_B = "zip_code"
display(con.sql("DESCRIBE table_a").df(), con.sql("DESCRIBE table_b").df())
con.sql("""
    SELECT zip_code, count(*) AS n
    FROM table_b
    GROUP BY zip_code
    HAVING count(*) > 1
""")


,column_name,column_type,null,key,default,extra
0,unique_key,VARCHAR,YES,None,None,None
1,created_date,TIMESTAMP_NS,YES,None,None,None
2,closed_date,TIMESTAMP_NS,YES,None,None,None
3,agency,VARCHAR,YES,None,None,None
4,complaint_type,VARCHAR,YES,None,None,None
5,zip_code,VARCHAR,YES,None,None,None
6,borough,VARCHAR,YES,None,None,None
7,status,VARCHAR,YES,None,None,None
8,closed_before_created,BOOLEAN,YES,None,None,None


,column_name,column_type,null,key,default,extra
0,zip_code,VARCHAR,YES,None,None,None
1,population,BIGINT,YES,None,None,None
2,population_moe,BIGINT,YES,None,None,None
3,estimate_period,VARCHAR,YES,None,None,None


┌──────────┬───────┐
│ zip_code │   n   │
│ varchar  │ int64 │
├──────────┴───────┤
│      0 rows      │
└──────────────────┘

## B4. Join and count

In [24]:
rows_before = con.sql("SELECT count(*) FROM table_a").fetchone()[0]
rows_inner = con.sql("SELECT count(*) FROM table_a a JOIN table_b b ON a.zip_code = b.zip_code").fetchone()[0]
rows_left = con.sql("SELECT count(*) FROM table_a a LEFT JOIN table_b b ON a.zip_code = b.zip_code").fetchone()[0]
print("Main rows before join:", rows_before)
print("Lookup rows:", len(b))
print("Rows after INNER JOIN:", rows_inner)
print("Rows after LEFT JOIN:", rows_left)
assert rows_left == rows_before, "The lookup join unexpectedly multiplied rows"
con.sql("""
    SELECT count(*) AS unmatched_requests,
           count(*) FILTER (WHERE a.zip_code IS NULL) AS missing_zip_requests,
           count(*) FILTER (WHERE a.zip_code IS NOT NULL) AS nonmatching_zip_requests
    FROM table_a a
    LEFT JOIN table_b b ON a.zip_code = b.zip_code
    WHERE b.zip_code IS NULL
""")


Main rows before join: 323630
Lookup rows: 218
Rows after INNER JOIN: 320500
Rows after LEFT JOIN: 323630


┌────────────────────┬──────────────────────┬──────────────────────────┐
│ unmatched_requests │ missing_zip_requests │ nonmatching_zip_requests │
│       int64        │        int64         │          int64           │
├────────────────────┼──────────────────────┼──────────────────────────┤
│               3130 │                 2973 │                      157 │
└────────────────────┴──────────────────────┴──────────────────────────┘

## B5. Find what didn't match

In [25]:
# Return all unmatched codes, including NULL; do not hide the last groups with LIMIT 20.
unmatched = con.sql("""
    SELECT a.zip_code, count(*) AS unmatched_rows
    FROM table_a a
    LEFT JOIN table_b b ON a.zip_code = b.zip_code
    WHERE b.zip_code IS NULL
    GROUP BY a.zip_code
    ORDER BY unmatched_rows DESC, a.zip_code
""").df()
unmatched


,zip_code,unmatched_rows
0,None,2973
1,10121,48
2,10000,32
3,10118,11
4,10281,11
5,10105,8
6,10179,6
7,10158,5
8,10041,4
9,10107,4


**What did not match, and what will I do about it?**

The main table has 323,630 rows and the population lookup has 218 rows with unique, non-null keys. The INNER JOIN returns 320,500 rows; the LEFT JOIN retains all 323,630. The 3,130 unmatched requests consist of 2,973 missing ZIP codes and 157 requests spread across 24 other ZIP codes. For example, 10000 has no match. It could be a placeholder or invalid code; some other discrepancies may result from ZIP/ZCTA differences, so I will not guess replacement codes.

I retain every request in the cleaned main table and the diagnostic LEFT JOIN. The rate calculation below uses only matching codes with positive population, and reports the excluded rows separately. The lookup was selected from codes present in Lab 1, so this join cannot identify every NYC area with zero complaints.


## B6. Answer one question that needs both tables

Write a query that joins the tables and uses columns from **both**, e.g. average GDP by region, complaints per 1,000 residents by ZIP code.

In [26]:
# Question: among matched postal areas with positive population, which have
# the most September 2026 requests per 1,000 estimated residents?
rates = con.sql("""
    SELECT b.zip_code,
           b.population,
           b.population_moe,
           b.estimate_period,
           count(*) AS request_count,
           round(count(*) * 1000.0 / NULLIF(b.population, 0), 2) AS requests_per_1000
    FROM table_a a
    JOIN table_b b ON a.zip_code = b.zip_code
    WHERE b.population > 0
    GROUP BY b.zip_code, b.population, b.population_moe, b.estimate_period
    ORDER BY requests_per_1000 DESC, b.zip_code
    LIMIT 10
""").df()
display(rates)
excluded = con.sql("""
    SELECT CASE WHEN b.zip_code IS NULL THEN 'unmatched_or_missing_zip'
                WHEN b.population = 0 THEN 'zero_population'
                ELSE 'included_positive_population' END AS category,
           count(*) AS requests
    FROM table_a a LEFT JOIN table_b b ON a.zip_code = b.zip_code
    GROUP BY category ORDER BY category
""").df()
display(excluded)
assert int(excluded["requests"].sum()) == len(a)
(ROOT / "review").mkdir(exist_ok=True)
rates.to_csv(ROOT / "review/lab2_top_rates.csv", index=False)
excluded.to_csv(ROOT / "review/lab2_join_disposition.csv", index=False)
unmatched.to_csv(ROOT / "review/lab2_unmatched.csv", index=False)


,zip_code,population,population_moe,estimate_period,request_count,requests_per_1000
0,11430,588,45,2020-2024,450,765.31
1,10152,7,12,2020-2024,1,142.86
2,11366,13463,1378,2020-2024,1875,139.27
3,11239,16410,1562,2020-2024,1990,121.27
4,10004,3711,909,2020-2024,422,113.72
5,10007,7406,1151,2020-2024,743,100.32
6,10012,21426,1506,2020-2024,1795,83.78
7,10018,8559,1139,2020-2024,680,79.45
8,10036,31036,2192,2020-2024,2406,77.52
9,10464,3828,753,2020-2024,294,76.80


,category,requests
0,included_positive_population,320324
1,unmatched_or_missing_zip,3130
2,zero_population,176


**What does the result tell us?**

Among matched areas with positive population, 11430 has the highest rate: 450 requests divided by 588 estimated residents gives 765.31 requests per 1,000. The next area, 10152, has only 7 estimated residents and a population margin of error of 12, so even one request gives a rate of 142.86. These small denominators make a simple ranking unstable and potentially misleading; they should not be described as the areas with the most dissatisfied residents.

The rate table includes 320,324 requests. It excludes 3,130 requests without a population match and 176 matching requests in the 22 zero-population ZCTAs; all remain in the cleaned source table. Requests are not unique residents, postal ZIPs only approximate ZCTAs, and September 2026 counts are divided by 2020–2024 estimates. These are descriptive rates, not a direct measure of incident prevalence or resident dissatisfaction.

## B7. Review before submission
Check that every cell ran, inspect the join exclusions and rate results, review the reflection in labs/lab2.md, and save the notebook. This review package has not been committed, pushed or submitted.


In [27]:
con.close()
print("Review copy executed. No GitHub push or Canvas submission was performed.")


Review copy executed. No GitHub push or Canvas submission was performed.
